In [ ]:
print("""
CERN Electron Collision Sensor Data Analysis
Notebook 9: Sensor Value Prediction
Objectives:
1. Identify redundant properties using physical relationships
2. Train a machine learning model to predict missing sensor values using remaining sensors
3. Evaluate recovery accuracy of the reconstructed physical variables
""")

In [ ]:
# Import required libraries
import warnings
warnings.filterwarnings('ignore')
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
print("Libraries loaded successfully")

In [ ]:
# Load engineered dataset and inspect redundancy
df = pd.read_csv('../data/processed/dielectron_engineered.csv')
df.columns = df.columns.str.strip()
print("Dataset shape:", df.shape)

In [ ]:
# Verify the physical redundancy of transverse momentum pt
# Formula: pt = sqrt(px^2 + py^2)
pt2_calculated = np.sqrt(df['px2']**2 + df['py2']**2)
pt2_error = np.abs(df['pt2'] - pt2_calculated)
print("Mean difference between measured pt2 and calculated pt2:", pt2_error.mean())

In [ ]:
# Train a model to predict total energy E2 using only other sensor data
# Features to exclude: E2, Total_Energy, Energy_Difference, Energy_Ratio, M, M_reco
exclude_cols = ['E2', 'Total_Energy', 'Energy_Difference', 'Energy_Ratio', 'M', 'M_reco']
X_e2 = df.drop(columns=exclude_cols)
y_e2 = df['E2']
X_train, X_test, y_train, y_test = train_test_split(X_e2, y_e2, test_size=0.20, random_state=42)
print(f"Features used for predicting E2: {X_e2.columns.tolist()}")

In [ ]:
# Train a fast Ridge Regressor to predict E2
ridge_model = Ridge()
ridge_model.fit(X_train, y_train)
pred_e2_ridge = ridge_model.predict(X_test)
print("Ridge regressor trained")

In [ ]:
# Train a fast Random Forest Regressor to predict E2 on a sample
rf_predict = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1)
# We sample training set for faster execution
X_train_sample = X_train.sample(n=10000, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]
rf_predict.fit(X_train_sample, y_train_sample)
pred_e2_rf = rf_predict.predict(X_test)
print("Random Forest regressor trained on sample")

In [ ]:
# Predict using physical reconstruction formula: E2_physics = sqrt(px2^2 + py2^2 + pz2^2)
pred_e2_physics = np.sqrt(X_test['px2']**2 + X_test['py2']**2 + X_test['pz2']**2)
print("Physics reconstruction computed")

In [ ]:
# Compare performance of ML models vs. pure physics reconstruction
r2_ridge = r2_score(y_test, pred_e2_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, pred_e2_ridge))
r2_rf = r2_score(y_test, pred_e2_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_e2_rf))
r2_physics = r2_score(y_test, pred_e2_physics)
rmse_physics = np.sqrt(mean_squared_error(y_test, pred_e2_physics))

results_df = pd.DataFrame({
    'Method': ['Ridge Regression', 'Random Forest Regressor', 'Physics Calculator (Redundancy)'],
    'RMSE': [rmse_ridge, rmse_rf, rmse_physics],
    'R2': [r2_ridge, r2_rf, r2_physics]
})
display(results_df)

In [ ]:
# Plot actual vs predicted values for E2
plt.figure(figsize=(7, 7))
plt.scatter(y_test, pred_e2_rf, alpha=0.3, label='Random Forest', color='blue')
plt.scatter(y_test, pred_e2_physics, alpha=0.3, label='Physics Calculator', color='green', marker='x')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Reference Line')
plt.title("Comparison of Energy E2 Reconstruction Methods")
plt.xlabel("Actual Energy E2 (GeV)")
plt.ylabel("Predicted Energy E2 (GeV)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save prediction results to outputs
os.makedirs('../outputs', exist_ok=True)
results_df.to_csv('../outputs/sensor_prediction_results.csv', index=False)
print("Sensor prediction results saved successfully")